# Notebook 02 — Feature Engineering
**Project:** Medical Insurance Cost Prediction  
**Goal:** Transform raw data into a model-ready feature matrix.

---
### Flow
1. Load raw data
2. Handle duplicates & data types
3. Encode categorical features
4. Create engineered features (BMI category, age group, smoker×obese)
5. Outlier analysis
6. Final feature matrix inspection
7. Save processed dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
print('Raw shape:', df.shape)
df.head()

## 1. Handle Duplicates

In [ ]:
print(f'Duplicates found: {df.duplicated().sum()}')
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after removing duplicates: {df.shape}')

## 2. Encode Categorical Features

In [ ]:
df_enc = df.copy()

# Binary encoding
df_enc['sex']    = df_enc['sex'].map({'male': 1, 'female': 0})
df_enc['smoker'] = df_enc['smoker'].map({'yes': 1, 'no': 0})

# One-hot encode region
df_enc = pd.get_dummies(df_enc, columns=['region'], drop_first=True)

print('Encoded columns:', df_enc.columns.tolist())
df_enc.head()

## 3. Engineered Features

Based on EDA findings, we create three new features:
- **`bmi_category`** — WHO BMI classification (Underweight / Normal / Overweight / Obese)
- **`is_obese`** — Binary flag: BMI >= 30
- **`age_group`** — Life stage buckets
- **`smoker_obese`** — Interaction term: the highest-cost segment identified in EDA

In [ ]:
# BMI Category (WHO classification)
df_enc['bmi_category'] = pd.cut(
    df_enc['bmi'],
    bins=[0, 18.5, 24.9, 29.9, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
).astype(str)

# Obese flag
df_enc['is_obese'] = (df_enc['bmi'] >= 30).astype(int)

# Age groups
df_enc['age_group'] = pd.cut(
    df_enc['age'],
    bins=[17, 30, 45, 60, float('inf')],
    labels=['Young (18-30)', 'Adult (31-45)', 'Middle-Aged (46-60)', 'Senior (60+)']
).astype(str)

# KEY: Smoker x Obese interaction
df_enc['smoker_obese'] = df_enc['smoker'] * df_enc['is_obese']

print('New features added: bmi_category, is_obese, age_group, smoker_obese')
df_enc[['age', 'bmi', 'bmi_category', 'is_obese', 'age_group', 'smoker', 'smoker_obese', 'charges']].head(10)

In [ ]:
# Visualize: engineered features vs charges
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BMI Category vs Charges
bmi_order = ['Underweight', 'Normal', 'Overweight', 'Obese']
bmi_means = df.copy()
bmi_means['bmi_category'] = pd.cut(
    bmi_means['bmi'], bins=[0, 18.5, 24.9, 29.9, float('inf')],
    labels=bmi_order
).astype(str)
grp = bmi_means.groupby('bmi_category')['charges'].mean().reindex(bmi_order)
axes[0].bar(grp.index, grp.values, color='steelblue', edgecolor='white')
axes[0].set_title('Avg Charges by BMI Category')
axes[0].set_xlabel('BMI Category')
axes[0].set_ylabel('Avg Charges ($)')
for i, v in enumerate(grp.values):
    axes[0].text(i, v + 100, f'${v:,.0f}', ha='center', fontsize=9)

# Smoker x Obese interaction
df['smoker_obese_label'] = df.apply(
    lambda r: f"Smoker+Obese" if r['smoker']=='yes' and r['bmi']>=30
    else ("Smoker" if r['smoker']=='yes'
    else ("Obese" if r['bmi']>=30 else "Neither")), axis=1
)
order = ['Neither', 'Obese', 'Smoker', 'Smoker+Obese']
grp2 = df.groupby('smoker_obese_label')['charges'].mean().reindex(order)
colors = ['steelblue', 'seagreen', 'orange', 'tomato']
axes[1].bar(grp2.index, grp2.values, color=colors, edgecolor='white')
axes[1].set_title('Avg Charges: Smoker x Obese Interaction')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Avg Charges ($)')
for i, v in enumerate(grp2.values):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontsize=9)

plt.suptitle('Engineered Features vs Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/07_engineered_features.png', bbox_inches='tight')
plt.show()

## 4. Outlier Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['age', 'bmi', 'charges']):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(f'{col.title()} — Outlier Check')
    ax.set_ylabel(col)

plt.suptitle('Outlier Detection (Box Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/08_outliers.png', bbox_inches='tight')
plt.show()

# IQR outlier counts
for col in ['age', 'bmi', 'charges']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'{col:10s}: {outliers} outliers (keeping — they represent genuine high-risk cases)')

## 5. Final Feature Matrix

In [ ]:
# Final columns for modelling (drop label-only columns)
model_cols = [c for c in df_enc.columns if c not in ['bmi_category', 'age_group']]
df_final = df_enc[model_cols]

print('Final feature matrix shape:', df_final.shape)
print('\nFeatures used in model:')
for c in df_final.columns:
    print(f'  {c}')
df_final.head()

In [ ]:
# Save processed data
from src.preprocessing import preprocess
df_clean = preprocess(load_raw_data(), save=True)
print('Processed data saved to data/processed/insurance_clean.csv')

## Summary

| Step | Action | Result |
|---|---|---|
| Duplicates | Removed 1 row | 1337 clean rows |
| sex / smoker | Binary encoded | 0/1 integers |
| region | One-hot encoded | 3 dummy columns |
| bmi_category | WHO classification | Underweight/Normal/Overweight/Obese |
| is_obese | BMI >= 30 flag | Binary 0/1 |
| age_group | Life stage buckets | 4 age bands |
| smoker_obese | smoker × is_obese | Key interaction feature |

> **Next step:** `03_Model_Training.ipynb` — train 5 regression models on this feature matrix and compare performance.